In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

google_api_key = os.getenv("GOOGLE_API_KEY")
if not google_api_key:
    raise ValueError("GOOGLE_API_KEY is missing in the .env file.")

with open("text.txt", "r", encoding="utf-8") as file:
    context = file.read()

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=google_api_key
)

question = input("Ask: ")
response = llm.invoke(
    f"""Context:{context}\nQuestion:{question}\nAnswer based only on the context."""
)

print(response.content)


In [ ]:
import os
from dotenv import load_dotenv
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS

load_dotenv()

google_api_key = os.getenv("GOOGLE_API_KEY")
if not google_api_key:
    raise ValueError("GOOGLE_API_KEY is missing in the .env file.")

with open("text.txt", "r", encoding="utf-8") as file:
    text = file.read()

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
documents = splitter.create_documents([text])

embeddings = GoogleGenerativeAIEmbeddings(
    model=os.getenv("GOOGLE_EMBEDDING_MODEL", "models/text-embedding-004"),
    google_api_key=google_api_key
)

db = FAISS.from_documents(documents, embeddings)
question = input("Ask: ")
retrieved_docs = db.similarity_search(question, k=2)
context = "\n\n".join(doc.page_content for doc in retrieved_docs)

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=google_api_key
)

prompt = f"""Answer the question using only the context below.
If the answer is not in the context, say that it is not available in the provided text.

Context:
{context}

Question: {question}
Answer:"""

response = llm.invoke(prompt)
print("Retrieved context:\n", context)
print("\nAnswer:\n", response.content)
